# Boosting, Bagging y Stacking

In [59]:
import numpy as np    # cálculo numérico en Python
import pandas as pd
from sklearn.datasets import fetch_openml   # cargar datasets desde el repositorio OpenML
from sklearn.model_selection import train_test_split    # divide un conjunto de datos en dos subconjuntos: uno para entrenamiento y otro para pruebas, de manera aleatoria.
from sklearn.metrics import accuracy_score    # calcula la precisión de un clasificador, que es la proporción de predicciones correctas sobre el total de predicciones.
from sklearn.tree import DecisionTreeClassifier   # divide los datos en subconjuntos basándose en las características del dataset
from sklearn.linear_model import LogisticRegression   #  clasificación binaria (sigmoide)

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import BaggingClassifier

# Random Forest Classifier
---
Un `RandomForestClassifier` crea un bosque de árboles de decisión. Cada árbol es entrenado con un subconjunto diferente del conjunto de datos. La predicción final es el resultado de promediar (o votar) las predicciones de todos los árboles. Es robusto contra el sobreajuste y generalmente proporciona una buena precisión.

# Adaptive Boosting Classifier
---
`AdaBoostClassifier` (Adaptive Boosting) combina varios clasificadores "débiles" (normalmente árboles de decisión de un solo nivel, también conocidos como "stumps"). Entrena estos clasificadores de forma secuencial, ajustando los pesos de las instancias de entrenamiento en función de los errores cometidos por los clasificadores anteriores. Los clasificadores que siguen intentan corregir los errores de los anteriores.

# Gradient Boosting Classifier
---
`GradientBoostingClassifier` construye un modelo aditivo en una secuencia de predictores. Cada nuevo predictor corrige los errores cometidos por el conjunto de predictores anteriores. Es una técnica poderosa que puede lograr una gran precisión, aunque es más propensa al sobreajuste que los métodos de ensamble anteriores.

# Bootstrap Aggregating Classifier
---
`BaggingClassifier` (Bootstrap Aggregating) entrena múltiples modelos en diferentes subconjuntos del conjunto de datos de entrenamiento (creados mediante muestreo con reemplazo). Las predicciones se combinan por votación (para clasificación) o promediado (para regresión). Este método ayuda a reducir la varianza del modelo y puede mejorar la estabilidad y precisión.

In [60]:
# Cargar datos
datos = fetch_openml(name = "credit-g", version = 1, as_frame = True, parser = 'pandas')
X = datos.data    # separa las características (atributos) del dataset y las almacena en 'X'
y = (datos.target == "good").astype(int) # Conversión "good" a 1 y "bad" a 0
X

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,no checking,12,existing paid,furniture/equipment,1736,<100,4<=X<7,3,female div/dep/mar,none,4,real estate,31,none,own,1,unskilled resident,1,none,yes
996,<0,30,existing paid,used car,3857,<100,1<=X<4,4,male div/sep,none,4,life insurance,40,none,own,1,high qualif/self emp/mgmt,1,yes,yes
997,no checking,12,existing paid,radio/tv,804,<100,>=7,4,male single,none,4,car,38,none,own,1,skilled,1,none,yes
998,<0,45,existing paid,radio/tv,1845,<100,1<=X<4,4,male single,none,4,no known property,23,none,for free,1,skilled,1,yes,yes


Aquí se utiliza fetch_openml para cargar el conjunto de datos credit-g desde la plataforma OpenML. Este conjunto de datos contiene información sobre solicitudes de crédito y si fueron buenas ("good") o malas ("bad").

* name = "credit-g": Especifica el nombre del dataset a cargar.
* version = 1: Especifica la versión del dataset.
* as_frame = True: Indica que los datos deben ser devueltos como un DataFrame de pandas.
* parser = 'pandas': Usa pandas para parsear los datos, lo que es útil para manipular y analizar los datos fácilmente.

In [61]:
# Verificar si hay valores no numéricos en las características
print("Valores únicos en cada columna:")
for column in X.columns:
    print(f"{column}: {X[column].unique()}")

Valores únicos en cada columna:
checking_status: ['<0', '0<=X<200', 'no checking', '>=200']
Categories (4, object): ['0<=X<200', '<0', '>=200', 'no checking']
duration: [ 6 48 12 42 24 36 30 15  9 10  7 60 18 45 11 27  8 54 20 14 33 21 16  4
 47 13 22 39 28  5 26 72 40]
credit_history: ['critical/other existing credit', 'existing paid', 'delayed previously', 'no credits/all paid', 'all paid']
Categories (5, object): ['all paid', 'critical/other existing credit', 'delayed previously',
                         'existing paid', 'no credits/all paid']
purpose: ['radio/tv', 'education', 'furniture/equipment', 'new car', 'used car', 'business', 'domestic appliance', 'repairs', 'other', 'retraining']
Categories (10, object): ['domestic appliance', 'new car', 'used car', 'business', ..., 'other', 'radio/tv',
                          'repairs', 'retraining']
credit_amount: [ 1169  5951  2096  7882  4870  9055  2835  6948  3059  5234  1295  4308
  1567  1199  1403  1282  2424  8072 12579  3430 

In [62]:
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)
print(f"X_train data type: {X_train.dtypes}")
print(f"y_train data type: {y_train.dtypes}")

X_train data type: checking_status           category
duration                     int64
credit_history            category
purpose                   category
credit_amount                int64
savings_status            category
employment                category
installment_commitment       int64
personal_status           category
other_parties             category
residence_since              int64
property_magnitude        category
age                          int64
other_payment_plans       category
housing                   category
existing_credits             int64
job                       category
num_dependents               int64
own_telephone             category
foreign_worker            category
dtype: object
y_train data type: int64


In [63]:
# Transformar características categóricas a numéricas (one-hot encoding)
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# Alinear X_train y X_test para asegurarse de que tienen las mismas columnas
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [64]:
# Verificar que no hay valores no numéricos en X_train y X_test
print("Tipos de datos después de la transformación:")
print(X_train.dtypes)

Tipos de datos después de la transformación:
duration                  int64
credit_amount             int64
installment_commitment    int64
residence_since           int64
age                       int64
                          ...  
job_skilled                bool
own_telephone_none         bool
own_telephone_yes          bool
foreign_worker_no          bool
foreign_worker_yes         bool
Length: 61, dtype: object


In [70]:
# Copiar datos a un nuevo DataFrame
df = datos.copy()
df

{'data':     checking_status  duration                  credit_history  \
 0                <0         6  critical/other existing credit   
 1          0<=X<200        48                   existing paid   
 2       no checking        12  critical/other existing credit   
 3                <0        42                   existing paid   
 4                <0        24              delayed previously   
 ..              ...       ...                             ...   
 995     no checking        12                   existing paid   
 996              <0        30                   existing paid   
 997     no checking        12                   existing paid   
 998              <0        45                   existing paid   
 999        0<=X<200        45  critical/other existing credit   
 
                  purpose  credit_amount    savings_status  employment  \
 0               radio/tv           1169  no known savings         >=7   
 1               radio/tv           5951          

In [66]:
# Definir los clasificadores de base (débiles)
decision_tree = DecisionTreeClassifier(random_state=42)
logistic_regression = LogisticRegression(max_iter=1000, random_state=42)

# Definir clasificadores de ensamble
random_forest = RandomForestClassifier(n_estimators=50, random_state=42)    # n_estimators=50 indica que el bosque tendrá 50 árboles
adaboost = AdaBoostClassifier(estimator=decision_tree, n_estimators=50, random_state=42)    # n_estimators=50 indica que se usarán 50 clasificadores débiles
gradient_boosting = GradientBoostingClassifier(n_estimators=50, random_state=42)
bagging = BaggingClassifier(estimator=decision_tree, n_estimators=50, random_state=42)

In [67]:
# Entrenar un árbol de decisión
decision_tree.fit(X_train, y_train)
y_pred_tree = decision_tree.predict(X_test)
accuracy_tree = accuracy_score(y_test, y_pred_tree)
print(f"Precisión del Árbol de Decisión: {accuracy_tree:.2f}")

# Entrenar una regresión logística
logistic_regression = LogisticRegression(max_iter=1000, random_state=42)
logistic_regression.fit(X_train, y_train)
y_pred_logistic = logistic_regression.predict(X_test)
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
print(f"Precisión de la Regresión Logística: {accuracy_logistic:.2f}")

Precisión del Árbol de Decisión: 0.68
Precisión de la Regresión Logística: 0.77


In [68]:
# Lista para las métricas de cada modelo
accuracy_resultados = []

# Lista de clasificadores
classifiers = {
    'Decision Tree': decision_tree,
    'Logistic Regression': logistic_regression,
    'Random Forest': random_forest,
    'AdaBoost': adaboost,
    'Gradient Boosting': gradient_boosting,
    'Bagging': bagging
}

# Entrenar y evaluar cada clasificador
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_resultados.append((name, accuracy))
    print(f"{name}: {accuracy:.2f}")

# Imprimir resultados
print("Resultados de precisión para cada clasificador:")
for name, accuracy in accuracy_resultados:
    print(f"{name}: {accuracy:.2f}")

Decision Tree: 0.68
Logistic Regression: 0.77
Random Forest: 0.76
AdaBoost: 0.67
Gradient Boosting: 0.76
Bagging: 0.75
Resultados de precisión para cada clasificador:
Decision Tree: 0.68
Logistic Regression: 0.77
Random Forest: 0.76
AdaBoost: 0.67
Gradient Boosting: 0.76
Bagging: 0.75


In [69]:
# Realizar un stacking
# Re-entrenar los clasificadores en el conjunto de entrenamiento
for clf in classifiers.values():
    clf.fit(X_train, y_train)

# Generar predicciones con los clasificadores base para el conjunto de prueba
predicciones = np.column_stack([clf.predict(X_test) for clf in classifiers.values()])

# Definir el clasificador de stacking
stacking = LogisticRegression(max_iter=1000, random_state=42)
stacking.fit(predicciones, y_test)

# Evaluación de stacking
stacking_predicciones = stacking.predict(predicciones)
stacking_accuracy = accuracy_score(y_test, stacking_predicciones)
print(f"Stacking Clasificador Accuracy: {stacking_accuracy:.2f}")

Stacking Clasificador Accuracy: 0.76


Es importante asegurarse de que cada clasificador base esté entrenado antes de usarlos para generar predicciones para el modelo de stacking. Se usa `np.column_stack` para apilar las predicciones de cada clasificador base como columnas, creando un nuevo conjunto de características que será usado para entrenar el modelo de stacking.